# Session 9.2: Building ML APIs with Flask

**Learning Objectives:**
1. Understand REST APIs and HTTP methods
2. Build Flask API for ML predictions
3. Handle JSON input/output
4. Implement error handling and validation
5. Test API with Postman/curl

**Duration:** 2 hours (60 min hands-on)

---

## Part 1: Understanding REST APIs

**API (Application Programming Interface):** A way for programs to communicate

**REST (Representational State Transfer):** Architectural style for APIs

**Key Concepts:**
- **Endpoints:** URLs that accept requests (e.g., `/predict`, `/health`)
- **HTTP Methods:**
  - `GET`: Retrieve data (read-only)
  - `POST`: Send data (create/update)
  - `PUT`: Update data
  - `DELETE`: Remove data
- **JSON:** Standard format for data exchange
- **Status Codes:**
  - `200`: Success
  - `400`: Bad request (client error)
  - `500`: Server error

**Why APIs for ML?**
- Decouple model from application
- Multiple clients can use same model
- Easy to update model without changing apps
- Language-agnostic (any language can call API)

## Part 2: Installing Flask

First, install required packages:

In [ ]:
# Install Flask (run this in terminal or uncomment)
# !pip install flask flask-cors

import sys
print(f"Python version: {sys.version}")

# Import libraries
import json
import numpy as np
import pandas as pd
import joblib

# For testing the API
import requests

print("Libraries imported successfully!")

## Part 3: Creating a Simple ML Model

Let's create a spam detector model that we'll serve via API.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Sample spam/ham dataset
messages = [
    "Free money! Click here now!",
    "Hey, are we still meeting for lunch?",
    "Congratulations! You've won a million dollars!",
    "Can you send me the report by tomorrow?",
    "URGENT: Your account will be closed!",
    "Looking forward to seeing you at the conference",
    "Get rich quick! Limited time offer!",
    "Thanks for your help with the project",
    "You have been selected for a special prize!",
    "Meeting rescheduled to 3 PM",
    "Buy now and save 90%!!!",
    "Please review the attached document",
    "Claim your free gift today!",
    "Great job on the presentation",
    "Act now! This offer expires soon!",
    "Can we schedule a call next week?",
    "FREE!!! No credit card required!",
    "The deadline has been extended to Friday",
    "Click this link to verify your account",
    "Thank you for your purchase",
] * 50  # Duplicate to create larger dataset

# Labels: 1 for spam, 0 for ham (not spam)
labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0] * 50

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    messages, labels, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Create pipeline
spam_detector = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000, stop_words='english')),
    ('classifier', MultinomialNB())
])

# Train
spam_detector.fit(X_train, y_train)

# Evaluate
y_pred = spam_detector.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

In [ ]:
# Save the model
import os
os.makedirs('spam_detector_api', exist_ok=True)

joblib.dump(spam_detector, 'spam_detector_api/model.pkl')
print("Model saved to spam_detector_api/model.pkl")

# Test loading
loaded_model = joblib.load('spam_detector_api/model.pkl')
test_message = "Congratulations! You won!"
prediction = loaded_model.predict([test_message])[0]
probability = loaded_model.predict_proba([test_message])[0]

print(f"\nTest prediction: {'Spam' if prediction == 1 else 'Ham'}")
print(f"Probability: {probability[prediction]:.4f}")

## Part 4: Building the Flask API

Now let's create the Flask application. This code will be saved as `app.py`.

In [ ]:
# This is the Flask API code - save this to spam_detector_api/app.py

flask_code = '''
from flask import Flask, request, jsonify
from flask_cors import CORS
import joblib
import numpy as np
import os

# Create Flask app
app = Flask(__name__)
CORS(app)  # Enable CORS for all routes

# Load model at startup
MODEL_PATH = 'model.pkl'
model = None

try:
    model = joblib.load(MODEL_PATH)
    print(f"✓ Model loaded successfully from {MODEL_PATH}")
except Exception as e:
    print(f"✗ Error loading model: {str(e)}")


@app.route('/', methods=['GET'])
def home():
    """Welcome endpoint"""
    return jsonify({
        'message': 'Spam Detector API',
        'version': '1.0.0',
        'endpoints': {
            'POST /predict': 'Make spam predictions',
            'GET /health': 'Check API health',
            'GET /': 'This page'
        }
    })


@app.route('/health', methods=['GET'])
def health():
    """Health check endpoint"""
    return jsonify({
        'status': 'healthy',
        'model_loaded': model is not None
    })


@app.route('/predict', methods=['POST'])
def predict():
    """Spam prediction endpoint"""
    try:
        # Check if model is loaded
        if model is None:
            return jsonify({
                'error': 'Model not loaded'
            }), 500
        
        # Get JSON data from request
        data = request.get_json()
        
        # Validate input
        if not data or 'text' not in data:
            return jsonify({
                'error': 'Missing required field: text'
            }), 400
        
        text = data['text']
        
        # Validate text is not empty
        if not text or not text.strip():
            return jsonify({
                'error': 'Text cannot be empty'
            }), 400
        
        # Make prediction
        prediction = model.predict([text])[0]
        probabilities = model.predict_proba([text])[0]
        
        # Prepare response
        result = {
            'text': text,
            'is_spam': bool(prediction == 1),
            'label': 'spam' if prediction == 1 else 'ham',
            'confidence': float(probabilities[prediction]),
            'probabilities': {
                'ham': float(probabilities[0]),
                'spam': float(probabilities[1])
            }
        }
        
        return jsonify(result), 200
    
    except Exception as e:
        return jsonify({
            'error': f'Prediction failed: {str(e)}'
        }), 500


if __name__ == '__main__':
    # Run the app
    port = int(os.environ.get('PORT', 5000))
    app.run(host='0.0.0.0', port=port, debug=True)
'''

# Save the Flask app
with open('spam_detector_api/app.py', 'w') as f:
    f.write(flask_code)

print("Flask app saved to spam_detector_api/app.py")

## Part 5: Creating Requirements File

In [ ]:
requirements = """flask==2.3.0
flask-cors==4.0.0
scikit-learn==1.3.0
numpy==1.24.0
joblib==1.3.0
gunicorn==21.2.0
"""

with open('spam_detector_api/requirements.txt', 'w') as f:
    f.write(requirements)

print("Requirements file created: spam_detector_api/requirements.txt")

## Part 6: Creating Test Script

Let's create a script to test our API.

In [ ]:
test_api_code = '''#!/usr/bin/env python3
"""
Test script for Spam Detector API

Usage:
    python test_api.py
"""

import requests
import json

# API base URL
BASE_URL = 'http://localhost:5000'


def test_home():
    """Test home endpoint"""
    print("\n" + "="*50)
    print("Testing GET /")
    print("="*50)
    
    response = requests.get(f"{BASE_URL}/")
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")


def test_health():
    """Test health endpoint"""
    print("\n" + "="*50)
    print("Testing GET /health")
    print("="*50)
    
    response = requests.get(f"{BASE_URL}/health")
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")


def test_predict(text):
    """Test prediction endpoint"""
    print("\n" + "="*50)
    print(f"Testing POST /predict")
    print(f"Text: {text}")
    print("="*50)
    
    payload = {'text': text}
    response = requests.post(
        f"{BASE_URL}/predict",
        json=payload,
        headers={'Content-Type': 'application/json'}
    )
    
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")


def test_error_cases():
    """Test error handling"""
    print("\n" + "="*50)
    print("Testing Error Cases")
    print("="*50)
    
    # Test 1: Missing text field
    print("\n1. Missing 'text' field:")
    response = requests.post(
        f"{BASE_URL}/predict",
        json={},
        headers={'Content-Type': 'application/json'}
    )
    print(f"Status: {response.status_code}")
    print(f"Response: {response.json()}")
    
    # Test 2: Empty text
    print("\n2. Empty text:")
    response = requests.post(
        f"{BASE_URL}/predict",
        json={'text': ''},
        headers={'Content-Type': 'application/json'}
    )
    print(f"Status: {response.status_code}")
    print(f"Response: {response.json()}")


if __name__ == '__main__':
    print("\n" + "#"*50)
    print("# Spam Detector API Test Suite")
    print("#"*50)
    
    try:
        # Test basic endpoints
        test_home()
        test_health()
        
        # Test predictions
        test_predict("Free money! Click here now!")
        test_predict("Hey, are we still meeting for lunch?")
        test_predict("Congratulations! You won a million dollars!")
        test_predict("Can you send me the report?")
        
        # Test error handling
        test_error_cases()
        
        print("\n" + "#"*50)
        print("# All tests completed!")
        print("#"*50 + "\n")
        
    except requests.exceptions.ConnectionError:
        print("\n✗ Error: Could not connect to API")
        print("Make sure the Flask app is running:")
        print("  python app.py")
    except Exception as e:
        print(f"\n✗ Error: {str(e)}")
'''

with open('spam_detector_api/test_api.py', 'w') as f:
    f.write(test_api_code)

print("Test script saved to spam_detector_api/test_api.py")

## Part 7: Creating README

In [ ]:
readme = """# Spam Detector API

A simple Flask-based REST API for spam detection using machine learning.

## Features

- Spam/Ham classification using Naive Bayes
- TF-IDF text vectorization
- Probability scores for predictions
- Error handling and input validation
- Health check endpoint

## Installation

1. Install dependencies:
```bash
pip install -r requirements.txt
```

2. Make sure `model.pkl` is in the same directory

## Running the API

```bash
python app.py
```

The API will start on `http://localhost:5000`

## API Endpoints

### GET /
Welcome page with API information

### GET /health
Health check endpoint

**Response:**
```json
{
  "status": "healthy",
  "model_loaded": true
}
```

### POST /predict
Make spam predictions

**Request:**
```json
{
  "text": "Congratulations! You won!"
}
```

**Response:**
```json
{
  "text": "Congratulations! You won!",
  "is_spam": true,
  "label": "spam",
  "confidence": 0.95,
  "probabilities": {
    "ham": 0.05,
    "spam": 0.95
  }
}
```

## Testing

Run the test script:
```bash
python test_api.py
```

Or use curl:
```bash
# Test prediction
curl -X POST http://localhost:5000/predict \\
  -H "Content-Type: application/json" \\
  -d '{"text": "Free money now!"}'

# Health check
curl http://localhost:5000/health
```

## Example Usage (Python)

```python
import requests

url = 'http://localhost:5000/predict'
data = {'text': 'Congratulations! You won!'}

response = requests.post(url, json=data)
result = response.json()

print(f"Is spam: {result['is_spam']}")
print(f"Confidence: {result['confidence']:.2%}")
```

## Error Handling

The API returns appropriate HTTP status codes:
- `200`: Successful request
- `400`: Bad request (missing/invalid input)
- `500`: Server error

## License

MIT
"""

with open('spam_detector_api/README.md', 'w') as f:
    f.write(readme)

print("README created: spam_detector_api/README.md")

## Part 8: Running and Testing the API

To run the API:

1. Open a terminal
2. Navigate to the `spam_detector_api` directory
3. Run: `python app.py`
4. In another terminal, run: `python test_api.py`

**Or test with curl:**

```bash
# Health check
curl http://localhost:5000/health

# Spam prediction
curl -X POST http://localhost:5000/predict \
  -H "Content-Type: application/json" \
  -d '{"text": "Free money! Click now!"}'

# Ham prediction  
curl -X POST http://localhost:5000/predict \
  -H "Content-Type: application/json" \
  -d '{"text": "Meeting at 3pm tomorrow"}'
```

## Summary & Key Takeaways

### What We Built:
1. **Complete Flask API** with:
   - Prediction endpoint (`POST /predict`)
   - Health check endpoint (`GET /health`)
   - Welcome page (`GET /`)

2. **Production Features:**
   - Input validation
   - Error handling
   - JSON request/response
   - CORS enabled
   - Proper HTTP status codes

3. **Testing Tools:**
   - Automated test script
   - curl examples
   - Complete documentation

### Best Practices:
- ✅ Load model once at startup (not per request)
- ✅ Validate all inputs
- ✅ Return meaningful error messages
- ✅ Use appropriate HTTP status codes
- ✅ Include health check endpoint
- ✅ Document API endpoints
- ✅ Enable CORS for web apps

### Next Session:
We'll learn FastAPI - a modern, faster alternative to Flask!

---

**Assignment:**
1. Run the spam detector API locally
2. Test with the provided test script
3. Try adding a new endpoint (e.g., `/batch-predict` for multiple texts)
4. Use Postman to test the API

**Estimated Time:** 45-60 minutes